# Training LoRA Qwen untuk KKA
Jalankan di Google Colab dengan GPU. Mulai dari PDF. Data demo hanya menguji alur, bukan dataset penelitian.
Versi Torch mengikuti runtime Colab; paket inti dipin dan versinya direkam.

## 1. Pilih GPU
Runtime > Change runtime type > GPU. T4 16 GB dapat menjadi titik awal jika tersedia. Ketersediaan dan durasi tidak dijamin.

In [ ]:
from google.colab import files
import zipfile
uploaded = files.upload()  # pilih Kode_Qwen_AstraDB_LoRA.zip
package = next(iter(uploaded))
with zipfile.ZipFile(package) as z:
    z.extractall('/content')


In [ ]:
%pip install -r /content/kka_lab/requirements-core.txt


## 2. Restart session
Setelah instalasi, pilih Runtime > Restart session. Lanjutkan dari sel berikut ini; tidak perlu mengunggah ZIP lagi selama berkas /content masih ada. Jika runtime diganti/dihapus, ulangi unggah paket.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import torch
print(torch.__version__, torch.version.cuda)
assert torch.cuda.is_available(), 'GPU belum aktif'
print(torch.cuda.get_device_name(0))
!nvidia-smi


## 3. Siapkan data dan record model di Drive
Buat MyDrive/kka_project/data/lora_dialogues.jsonl dan MyDrive/kka_project/model_record.json. Record harus hasil download_model.py di laptop. Jangan unggah token Astra ke notebook ini. File data memuat contoh dua mode pada setiap group_id.

In [ ]:
from pathlib import Path
project = Path('/content/drive/MyDrive/kka_project')
assert (project/'data/lora_dialogues.jsonl').exists(), 'Dataset belum ada'
assert (project/'model_record.json').exists(), 'Record model belum ada'
%cd /content/kka_lab


## 4. Latih adaptor
Hasil ditulis ke Drive. Jika contoh terlalu panjang, program akan menyebut ID yang perlu dipersingkat; jawaban tidak dipotong diam-diam. Training awal dua epoch bukan jaminan hasil terbaik.

In [ ]:
!python training/train_lora.py \
  --data /content/drive/MyDrive/kka_project/data/lora_dialogues.jsonl \
  --record /content/drive/MyDrive/kka_project/model_record.json \
  --out /content/drive/MyDrive/kka_project/adapters/run01


Jika sesi terputus, gunakan checkpoint yang benar-benar ada. Jalankan kembali perintah di atas dengan tambahan --resume /content/drive/MyDrive/kka_project/adapters/run01/checkpoints/checkpoint-N. Ganti N berdasarkan nama folder. Jangan resume jika data, model, atau konfigurasi telah diubah.

## 5. Unduh adaptor final
Unduh setelah training berhasil. ZIP berisi folder qwen-kka-lora. Tetap simpan run_manifest, packages.lock, evaluation, trainer_state, dan checkpoint di Drive untuk catatan penelitian.

In [ ]:
import shutil
from pathlib import Path
from google.colab import files
project = Path('/content/drive/MyDrive/kka_project')
final = project/'adapters/run01/final'
assert (final/'adapter_config.json').exists(), 'Training belum menghasilkan adaptor final'
stage = Path('/content/export_adapter')
stage.mkdir(exist_ok=True)
shutil.copytree(final, stage/'qwen-kka-lora', dirs_exist_ok=True)
archive = shutil.make_archive('/content/qwen-kka-lora', 'zip', stage)
files.download(archive)


## 6. Jalankan di laptop
Ekstrak ZIP ke folder adapters/ proyek lokal. Jalankan: python chat.py --mode terstruktur --adapter adapters/qwen-kka-lora. Uji juga mode umum memakai adaptor yang sama. Adaptor tidak menggantikan model dasar; keduanya dimuat bersama.